# 07 数值积分综合实验

本 Notebook 汇总第四章的关键实验：复合公式收敛阶、Romberg 求积表、自适应 Simpson 区间划分、Newton-Cotes 与 Gauss-Legendre 对比、带权积分、离散数据积分、二维区域积分和高维 Monte Carlo 积分。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    adaptive_simpson,
    average_parabolic_integral,
    closed_newton_cotes_weights,
    composite_simpson,
    composite_trapezoid,
    discrete_simpson,
    discrete_trapezoid,
    gauss_chebyshev_integrate,
    gauss_hermite_integrate,
    gauss_laguerre_integrate,
    gauss_legendre_integrate,
    monte_carlo_integrate,
    natural_cubic_spline_integral,
    newton_cotes_integrate,
    romberg_integrate,
    tensor_product_gauss_legendre,
    variable_limit_integral,
)

import integration_methods as chapter_script


## 实验 1：复合梯形和复合 Simpson 的收敛阶

对 $\sin x$ 在 $[0,\pi]$ 上积分，精确值为 2。理论上复合梯形是二阶，复合 Simpson 是四阶。


In [ ]:
script_rows = chapter_script.convergence_table()
print("脚本版快速检查：")
for row in script_rows:
    print(row)

n_values = np.array([8, 16, 32, 64, 128, 256])
exact = 2.0
trap_errors = np.array([abs(composite_trapezoid(np.sin, 0.0, math.pi, int(n)) - exact) for n in n_values])
simp_errors = np.array([abs(composite_simpson(np.sin, 0.0, math.pi, int(n)) - exact) for n in n_values])

plt.figure(figsize=(7, 4.0))
plt.loglog(n_values, trap_errors, "o-", label="复合梯形")
plt.loglog(n_values, simp_errors, "s-", label="复合 Simpson")
plt.xlabel("子区间数")
plt.ylabel("绝对误差")
plt.title("复合公式收敛阶实验")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 实验 2：Romberg 求积表

Romberg 表的第一列是复合梯形，后续列通过 Richardson 外推逐步消去 $h^2,h^4,\ldots$ 误差项。


In [ ]:
romberg = romberg_integrate(math.sin, 0.0, math.pi, max_order=6)
print(np.array2string(romberg.table, precision=12, suppress_small=False))
print("对角线误差:", np.abs(np.diag(romberg.table) - 2.0))


## 实验 3：自适应 Simpson 区间划分

对局部尖峰函数，自适应方法应把更多区间放在尖峰附近。


In [ ]:
peaked = lambda x: 1.0 / (1.0 + 400.0 * (x - 0.65) ** 2)
adaptive = adaptive_simpson(peaked, 0.0, 1.0, tolerance=1e-8, max_depth=24)
xx = np.linspace(0.0, 1.0, 1000)
yy = np.array([peaked(x) for x in xx])

plt.figure(figsize=(8, 4.2))
plt.plot(xx, yy, color="C0", label="被积函数")
for left, right in adaptive.intervals:
    plt.axvspan(left, right, color="C1", alpha=0.06)
plt.title("自适应 Simpson 接受区间")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

print("积分值:", adaptive.value)
print("区间数量:", adaptive.intervals.shape[0])
print("函数计算次数:", adaptive.evaluations)
print("是否未触及最大深度:", adaptive.converged)


## 实验 4：Newton-Cotes 与 Gauss-Legendre 对比

使用相近的函数计算次数比较 $e^x$ 在 $[-1,1]$ 上的误差。Gauss-Legendre 对光滑函数通常能用更少节点达到更高精度。


In [ ]:
exact_exp = math.e - 1.0 / math.e
node_counts = np.arange(2, 13)
nc_errors = []
gauss_errors = []
for nodes_count in node_counts:
    nodes, weights = closed_newton_cotes_weights(nodes_count - 1)
    nc_value = newton_cotes_integrate(lambda x: math.exp(x), -1.0, 1.0, nodes, weights)
    gauss_value = gauss_legendre_integrate(np.exp, -1.0, 1.0, int(nodes_count))
    nc_errors.append(abs(nc_value - exact_exp))
    gauss_errors.append(abs(gauss_value - exact_exp))

plt.figure(figsize=(7, 4.0))
plt.semilogy(node_counts, nc_errors, "o-", label="闭型 Newton-Cotes")
plt.semilogy(node_counts, gauss_errors, "s-", label="Gauss-Legendre")
plt.xlabel("函数计算次数")
plt.ylabel("绝对误差")
plt.title("等距节点与 Gauss 节点的比较")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 实验 5：典型带权积分

这些公式的重点不是普通积分，而是已经包含特定权函数的积分。


In [ ]:
weighted_results = {
    "Chebyshev x^2": (gauss_chebyshev_integrate(lambda x: x**2, 16), math.pi / 2),
    "Laguerre x": (gauss_laguerre_integrate(lambda x: x, 16), 1.0),
    "Hermite x^2": (gauss_hermite_integrate(lambda x: x**2, 16), math.sqrt(math.pi) / 2),
}
for name, (value, exact) in weighted_results.items():
    print(f"{name:16s} value={value:.12f}, exact={exact:.12f}, error={abs(value-exact):.1e}")


## 实验 6：离散数据积分

比较梯形、局部二次、平均抛物线和自然三次样条对采样数据的积分结果。


In [ ]:
rng = np.random.default_rng(2026)
x_data = np.linspace(0.0, math.pi, 31)
y_data = np.sin(x_data) + 0.02 * rng.normal(size=x_data.size)
discrete_methods = {
    "梯形": discrete_trapezoid(x_data, y_data),
    "Simpson": discrete_simpson(x_data, y_data),
    "平均抛物线": average_parabolic_integral(x_data, y_data),
    "自然三次样条": natural_cubic_spline_integral(x_data, y_data),
}
for name, value in discrete_methods.items():
    print(f"{name:10s} value={value:.10f}, error={abs(value-2.0):.3e}")

plt.figure(figsize=(7, 4.0))
plt.bar(discrete_methods.keys(), [abs(v - 2.0) for v in discrete_methods.values()])
plt.ylabel("绝对误差")
plt.title("离散含噪声数据积分误差")
plt.xticks(rotation=15)
plt.show()


## 实验 7：二维区域积分

矩形区域可用张量积求积；简单三角形区域可用变限积分。


In [ ]:
rect_value = tensor_product_gauss_legendre(lambda x, y: x + y, (0.0, 1.0), (0.0, 2.0), 5, 5)
triangle_area = variable_limit_integral(
    lambda x, y: np.ones_like(x),
    (0.0, 1.0),
    lambda x: np.zeros_like(x),
    lambda x: 1.0 - x,
    20,
    20,
)
print("矩形区域 int(x+y) dxdy:", rect_value, "exact", 3.0)
print("三角形面积:", triangle_area, "exact", 0.5)


## 实验 8：高维 Monte Carlo 积分

固定随机种子，报告样本量、估计值和标准误差。这里用 Gauss-Legendre 的一维高精度结果构造参考值。


In [ ]:
dimension = 8
bounds = np.array([[0.0, 1.0]] * dimension)
one_dim = gauss_legendre_integrate(lambda x: np.exp(-x**2), 0.0, 1.0, 80)
reference = one_dim**dimension
sample_sizes = np.array([1_000, 2_000, 5_000, 10_000, 20_000, 50_000])
mc_values = []
mc_errors = []
mc_standard_errors = []
for n in sample_sizes:
    result = monte_carlo_integrate(lambda points: np.exp(-np.sum(points**2, axis=1)), bounds, int(n), seed=2026)
    mc_values.append(result.value)
    mc_errors.append(abs(result.value - reference))
    mc_standard_errors.append(result.standard_error)
    print(f"N={n:<6d} value={result.value:.8f}, SE={result.standard_error:.3e}, error={abs(result.value-reference):.3e}")

plt.figure(figsize=(7, 4.0))
plt.loglog(sample_sizes, mc_errors, "o-", label="实际误差")
plt.loglog(sample_sizes, mc_standard_errors, "s-", label="标准误差")
plt.xlabel("样本量")
plt.ylabel("误差")
plt.title("高维 Monte Carlo 积分")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 综合小结

* 插值型求积固定节点后对插值多项式积分；Gauss 型求积利用正交多项式选择节点，从而提高代数精度。
* 复合方法通过局部化降低误差，外推方法利用误差展开消去主误差项，自适应方法根据局部误差重新分配计算资源。
* 对光滑低维函数，Gauss-Legendre 和 Romberg 往往很高效；对局部变化剧烈的函数，自适应方法更合适。
* 离散数据积分受到采样点和噪声限制，不能直接套用可自由取点的函数积分算法。
* 多重积分中张量积节点数随维数快速增长，高维问题通常需要随机方法、拟随机方法或问题结构的额外利用。
* 本章的误差阶、外推、自适应和加权积分思想会在后续数值微分、常微分方程和偏微分方程中继续出现。
